# Identification of activity Selectivity–Associated Genes

To identify genes associated with activity selectivity, we employed a multi-step analytical pipeline integrating single-cell–level screening and mouse-level consistency assessment.

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib as mpl
from datetime import date
import scanpy as sc
import omicverse as ov
from matplotlib_venn import venn3
import matplotlib.pyplot as plt
from pathlib import Path

import pickle as pkl

from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats
from pydeseq2.utils import load_example_data

SAVE = False  # whether to save the outputs of this notebook

In [ ]:
mpl.rcParams["font.family"] = "Arial"          
mpl.rcParams["font.sans-serif"] = ["Arial"]    
mpl.rcParams["pdf.fonttype"] = 42              
mpl.rcParams["ps.fonttype"]  = 42
mpl.rcParams["savefig.dpi"] = 300

mpl.rcParams["figure.dpi"]  = 300

In [ ]:
input_path = r"F:\Code\codex\PlaceCell_snRNAseq\Data\DEGs"
output_path = Path(r"F:\Code\codex\PlaceCell_snRNAseq\results")
output_path.mkdir(parents=True, exist_ok=True)
today = date.today().strftime("%Y%m%d")

df_ns = pd.read_csv(os.path.join(input_path,"SourceData_DEG_ActiveNonPlace_vs_Silent_Wilcoxon_CA1ProsOnly.csv"),index_col=0)
df_ps = pd.read_csv(os.path.join(input_path,"SourceData_DEG_Place_vs_Silent_Wilcoxon_CA1ProsOnly.csv"),index_col=0)
df_pn = pd.read_csv(os.path.join(input_path,"SourceData_DEG_Place_vs_ActiveNonPlace_Wilcoxon_CA1ProsOnly.csv"),index_col=0)

---

### 1. Exploratory DEG Screening at the Single-Cell Level

As an initial exploratory step, differential expression analysis was performed by treating individual cells as independent samples. Differentially expressed genes (DEGs) were identified using relaxed criteria (|log2FC| > 0.1 and unadjusted P value < 0.05). These permissive thresholds were intentionally applied to retain a broad pool of candidate genes for subsequent filtering and robustness analyses.

##### Definition of activity selectivity–associated genes

activity selectivity–associated genes were defined based on logical set operations across pairwise condition comparisons:
- Genes that were differentially expressed in both *place vs. non-place* and *place vs. silent* comparisons,
- Excluding genes detected exclusively in the *non-place vs. silent* comparison.

This definition aimed to prioritize genes specifically associated with place cell identity rather than differences attributable solely to neuronal activity.

##### Definition of activity-associated genes

Activity-associated genes were defined using complementary logical criteria:
- Genes that were differentially expressed in both *place vs. silent* and *non-place vs. silent* comparisons,
- Excluding genes identified exclusively in the *place vs. non-place* comparison.

This definition enriched for genes related to general neuronal activity independent of activity selectivity.

---


In [ ]:
df_pn.head()

In [ ]:
# Global DEG filtering parameters
P_COL  = "p_val"
LFC_COL = "avg_log2FC"

P_TH  = 0.05
LFC_TH = 0.1

In [ ]:
def filter_deg(
    df,
    direction="all",      # "all" | "up" | "down"
    p_th=P_TH,
    lfc_th=LFC_TH,
    p_col=P_COL,
    lfc_col=LFC_COL
):
    """
    Filter DEG table by p-value and direction.

    Parameters
    ----------
    direction : {"all", "up", "down"}
        all  : p < p_th
        up   : p < p_th & logFC > lfc_th
        down : p < p_th & logFC < -lfc_th
    """

    if direction == "all":
        q = f"{p_col} < @p_th and abs({lfc_col}) > @lfc_th"
    elif direction == "up":
        q = f"{p_col} < @p_th and {lfc_col} > @lfc_th"
    elif direction == "down":
        q = f"{p_col} < @p_th and {lfc_col} < -@lfc_th"
    else:
        raise ValueError("direction must be one of {'all','up','down'}")

    return df.query(q).copy()


In [ ]:
deg_ps_up = set(filter_deg(df_ps, direction="up").index)
deg_ps_down = set(filter_deg(df_ps, direction="down").index)

deg_ns_up = set(filter_deg(df_ns, direction="up").index)
deg_ns_down = set(filter_deg(df_ns, direction="down").index)

deg_pn_up = set(filter_deg(df_pn, direction="up").index)
deg_pn_down = set(filter_deg(df_pn, direction="down").index)

set_ps = deg_ps_up | deg_ps_down
set_ns = deg_ns_up | deg_ns_down
set_pn = deg_pn_up | deg_pn_down

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from upsetplot import UpSet
import inspect

# ================= 1. 整理多重集合与方向标签 =================
# (这部分代码与之前完全一致)
all_genes = set_ps | set_pn | set_ns
data = []
for g in all_genes:
    in_ps, in_pn, in_ns = g in set_ps, g in set_pn, g in set_ns
    is_up = (g in deg_ps_up) or (g in deg_pn_up) or (g in deg_ns_up)
    is_down = (g in deg_ps_down) or (g in deg_pn_down) or (g in deg_ns_down)
    
    if is_up and not is_down: direction = "Up"
    elif is_down and not is_up: direction = "Down"
    else: direction = "Discordant"
        
    data.append({
        "Active non-place vs Silent": in_ns, 
        "Place vs Active non-place": in_pn,  
        "Place vs Silent": in_ps,            
        "Direction": direction
    })

df = pd.DataFrame(data)
df = df.set_index(["Active non-place vs Silent", "Place vs Active non-place", "Place vs Silent"])

# ================= 2. 配置并绘制 UpSet 图 =================
kwargs = dict(
    show_counts=True, 
    sort_by="cardinality",
    intersection_plot_elements=0,
    # 【核心魔改 1】：把左侧的原生总数图彻底干掉
    totals_plot_elements=0 
)

sig = inspect.signature(UpSet.__init__).parameters
if "sort_categories_by" in sig: kwargs["sort_categories_by"] = None
elif "sort_sets_by" in sig: kwargs["sort_sets_by"] = None

fig = plt.figure(figsize=(11, 6)) # 稍微再加宽一点画布，给右侧留空间

upset = UpSet(df, **kwargs)
upset.add_stacked_bars(
    by="Direction",
    colors={"Up": "#E2A528", "Down": "#6FA3C6", "Discordant": "#d7d7d7"}, 
    title="Intersection size (genes)",
    elements=6,
)

axes = upset.plot(fig=fig)

# 处理堆叠柱状图底线


# ================= 3. 在右侧手动绘制总数图 =================
plt.subplots_adjust(right=0.75)

# 1) 先找到非空、且真的是 matplotlib Axes 的对象
valid_axes = [ax for ax in axes.values() if ax is not None and hasattr(ax, "get_legend")]

# 2) 找到当前带 legend 的轴
legend_ax = next((ax for ax in valid_axes if ax.get_legend() is not None), None)

# 3) 如果没找到，就退回到 stacked bar 轴（UpSet 常见）
if legend_ax is None:
    legend_ax = axes.get("extra1") or axes.get("intersections")

if legend_ax is not None:
    # 拿句柄和标签（比依赖 old legend 更稳）
    handles, labels = legend_ax.get_legend_handles_labels()

    # 删旧图例（若存在）
    old_leg = legend_ax.get_legend()
    if old_leg is not None:
        old_leg.remove()

    # 挂到 figure 级别，位置才稳定可控
    if handles:
        fig.legend(
            handles, labels,
            loc="upper left",
            bbox_to_anchor=(0.80, 0.90),  # 这里继续微调
            frameon=False
        )

# 获取中心点阵图 (matrix) 的位置
pos = axes['matrix'].get_position()

# 在矩阵右侧精准添加一个新的坐标轴区域 [left, bottom, width, height]
ax_right = fig.add_axes([pos.x1 + 0.03, pos.y0, 0.15, pos.height])

# 准备总数数据 (顺序要和矩阵的 Y 轴对应：最底端 -> 最顶端)
total_sizes = [len(set_ns), len(set_pn), len(set_ps)]

# 绘制水平条形图
bars = ax_right.barh([0, 1, 2], total_sizes, color='black', height=0.5)

# 美化右侧条形图
ax_right.set_yticks([]) # 隐藏 Y 轴刻度
ax_right.spines['top'].set_visible(False)
ax_right.spines['right'].set_visible(False)
ax_right.spines['left'].set_visible(False)
# ax_right.xaxis.grid(True, linestyle='--', alpha=0.5) # 加上辅助网格线
ax_right.set_xlabel("Set Size")

# 在柱子旁边写上具体数字 (可选)
for bar in bars:
    ax_right.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2, 
                  f'{int(bar.get_width())}', 
                  va='center', ha='left')
from matplotlib.patches import Patch

# 1) 找到真正的 stacked bar 轴
ax_stack = axes.get("extra1") or axes.get("intersections")
if ax_stack is None or len(getattr(ax_stack, "patches", [])) == 0:
    ax_stack = next(
        (ax for ax in axes.values()
         if ax is not None and ax.get_ylabel() == "Intersection size (genes)"),
        None
    )

# 2) 样式修复（网格/边框/底线）
if ax_stack is not None:
    ax_stack.grid(False, which="both", axis="both")
    for gl in ax_stack.get_xgridlines() + ax_stack.get_ygridlines():
        gl.set_visible(False)

    ax_stack.axhline(0, color="black", linewidth=1.0, clip_on=False, zorder=10)
    ax_stack.spines["bottom"].set_visible(False)
    ax_stack.spines["top"].set_visible(False)
    ax_stack.spines["right"].set_visible(False)
    ax_stack.spines["left"].set_visible(True)

# 3) 先删掉所有旧 legend（关键）
for ax in [a for a in axes.values() if a is not None and hasattr(a, "get_legend")]:
    leg = ax.get_legend()
    if leg is not None:
        leg.remove()


fig.canvas.draw_idle()
plt.savefig(os.path.join(output_path, f"Upset_lfc{LFC_COL}_{today}.png"), dpi = 300)
plt.show()

In [ ]:
activity_candidate_genes_all = (set_ps & set_ns) - set_pn

print(f"All activity genes:  {len(activity_candidate_genes_all)}")

In [ ]:
activity_candidate_df = pd.DataFrame({"gene": sorted(activity_candidate_genes_all)})

up_set = set(deg_ns_up)
down_set = set(deg_ns_down)

activity_candidate_df["label"] = np.select(
    [
        activity_candidate_df["gene"].isin(up_set) & ~activity_candidate_df["gene"].isin(down_set),
        activity_candidate_df["gene"].isin(down_set) & ~activity_candidate_df["gene"].isin(up_set),
        activity_candidate_df["gene"].isin(up_set) & activity_candidate_df["gene"].isin(down_set),
    ],
    ["Up", "Down", "Discordant"],   # 若不需要冲突类，可改成 pd.NA
    default=pd.NA
)

activity_candidate_df.head()

In [ ]:

activity_candidate_df.to_csv(
    os.path.join(
        output_path,
        f"activity_candidate_genes_all_lfc{LFC_TH}_{today}.csv"
    ),
    index=False
)


## 2. Mouse-level robustness and consistency analysis

To assess robustness across animals, we conducted a mouse-level consistency analysis. For each mouse, log2 fold changes were independently calculated in place vs non-place condition.

we demand 3/4 has similar trend

In [ ]:
adata = sc.read_h5ad(r"F:\Data\snRNAseq\PlaceCell\FilteredData\PCAnalysis_all_QC_20260127.h5ad")
adata_annotation = sc.read_h5ad(
    r"F:\Code\codex\PlaceCell_snRNAseq\Data\FilteredData\All_QC_filtered_annotation_20260604.h5ad"
)
assert adata.n_obs == adata_annotation.n_obs
assert (adata.obs_names == adata_annotation.obs_names).all()

# 转移 cell type 注释
adata.obs["cell_types"] = adata_annotation.obs["cell_types"].values

# 去掉 Unassigned
adata = adata[adata.obs["cell_types"] == "CA1-ProS"].copy()
adata

In [ ]:
adata_pn = adata[adata.obs["sample"].isin(["PC94", "PC110", "PC112", "PC114"])].copy()   #place vs nonplace
adata_ps = adata[adata.obs["sample"].isin(["PC103", "PC106", "PC118"])].copy() #place vs silent
adata_ns = adata[adata.obs["sample"].isin(["PC92", "PC105"])].copy()  #nonplace vs silent
# adata_pn.obs["condition"].value_counts()
# adata_ns = adata[adata.obs["sample"].isin(["PC92", "PC105", "PC132"])].copy()  #nonplace vs silent

In [ ]:
adata_sub = adata_ps

In [ ]:
adata_sub.obs["condition"].value_counts()

In [ ]:
adata_sub.obs["sample"].value_counts()

In [ ]:
samples = adata_sub.obs["sample"].unique()
conditions = adata_sub.obs["condition"].unique()

In [ ]:
pbs = []

for sample in samples:
    for condition in conditions:

        mask = (
            (adata_sub.obs["sample"] == sample) &
            (adata_sub.obs["condition"] == condition)
        )

        if mask.sum() == 0:
            raise ValueError(f"No cells for sample={sample}, condition={condition}")

        
        subset = adata_sub[mask].copy()
        subset.X = subset.layers["counts"]

        rep = sc.AnnData(
            X=subset.X.sum(axis=0, keepdims=True),
            var=subset.var
        )
        rep.obs_names = [f"{sample}_{condition}"]
        rep.obs["sample"] = sample
        rep.obs["condition"] = condition

        pbs.append(rep)


In [ ]:
pb = sc.concat(pbs)

In [ ]:
counts = pd.DataFrame(
    pb.X,
    index=pb.obs_names,   # ← 非常关键
    columns=pb.var_names
)
counts = counts.astype(int)
metadata = pb.obs[["sample", "condition"]]
assert (counts.index == metadata.index).all()


In [ ]:
((counts % 1) == 0).all().all()

In [ ]:
inference = DefaultInference(n_cpus=8)

log2fc_dict = {}   # key = mouse, value = pd.Series (genes)

for mouse in metadata["sample"].unique():

    # -------- 1. 拆分 counts / metadata（在建模之前！） --------
    mask = metadata["sample"] == mouse

    counts_m   = counts.loc[mask]        # (2 × n_genes)
    metadata_m = metadata.loc[mask]

    # 安全检查
    if counts_m.shape[0] != 2:
        print(f"Skip {mouse}: {counts_m.shape[0]} samples")
        continue

    # -------- 2. 构建 DeseqDataSet（只为 size factor） --------
    dds = DeseqDataSet(
        counts=counts_m,
        metadata=metadata_m,
        design_factors=["condition"],
        ref_level=["condition", "silent"],
        inference=inference,
    )

    dds.fit_size_factors()
    norm = dds.layers["normed_counts"]    # shape: (2, n_genes)

    # -------- 3. 明确行顺序，算 per-gene log2FC --------
    cond = metadata_m["condition"].values
    idx_place    = cond == "place"
    idx_nonplace = cond == "silent"

    log2fc = np.log2(
        (norm[idx_place, :].ravel() + 1) /
        (norm[idx_nonplace, :].ravel() + 1)
    )

    # -------- 4. 存成带基因名的 Series --------
    log2fc_dict[mouse] = pd.Series(
        log2fc,
        index=dds.var_names,
        name=mouse
    )

# -------- 5. 合并成 gene × mouse 的 DataFrame --------
log2fc_df = pd.concat(log2fc_dict, axis=1)

print(log2fc_df.shape)
# (n_genes, n_mice)


In [ ]:
# 1) 先取 Up/Down 基因（不必先转 list）
label_s = activity_candidate_df.set_index("gene")["label"]
up_genes = label_s.index[label_s.eq("Up")]
down_genes = label_s.index[label_s.eq("Down")]

n_up = up_genes.size
n_down = down_genes.size

# 2) 一次性计算一致性（避免重复写 subset 逻辑）
consistent_up = log2fc_df.reindex(up_genes).gt(0).sum(axis=1).ge(3)
consistent_down = log2fc_df.reindex(down_genes).lt(0).sum(axis=1).ge(3)

# 3) 如需最终一致基因列表
activity_candidate_genes_up_consistent = consistent_up.index[consistent_up].tolist()
activity_candidate_genes_down_consistent = consistent_down.index[consistent_down].tolist()

print(n_up, n_down)
print(len(activity_candidate_genes_up_consistent), len(activity_candidate_genes_down_consistent))


In [ ]:
# 1) 准备数据（和你前面一致）
label_s = activity_candidate_df.set_index("gene")["label"]
label_s = label_s[label_s.isin(["Up", "Down"])]

genes = label_s.index.intersection(log2fc_df.index)
sub = log2fc_df.loc[genes]

expected_sign = label_s.loc[genes].map({"Up": 1, "Down": -1})
is_consistent = sub.mul(expected_sign, axis=0).gt(0)   # 每个gene在每个样本是否方向一致
k_support = is_consistent.sum(axis=1)                  # 每个gene被多少个样本支持

tmp = pd.DataFrame({
    "k": k_support,
    "label": label_s.loc[genes].values
})

# 2) 统计每个k中 Up/Down 基因数
count_tbl = (
    tmp.groupby(["k", "label"]).size()
       .unstack(fill_value=0)
       .reindex(range(1, sub.shape[1] + 1), fill_value=0)
       .reindex(columns=["Up", "Down"], fill_value=0)
)

pct_tbl = count_tbl.div(count_tbl.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)

# 3) 画图
x = np.arange(len(count_tbl))
up_counts = count_tbl["Up"].values
down_counts = count_tbl["Down"].values
totals = up_counts + down_counts

fig, ax = plt.subplots(figsize=(4, 5))
b1 = ax.bar(x, up_counts, color="#E2A528", label="Up")
b2 = ax.bar(x, down_counts, bottom=up_counts, color="#6FA3C6", label="Down")

ax.set_xticks(x)
ax.set_xticklabels(count_tbl.index.astype(str))
ax.set_xlabel("Concordant samples")
ax.set_ylabel("Gene count")
ax.set_title("Up/Down composition by supporting-sample count")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# 4) 在柱内标注占比，在柱顶标注总数
for i in range(len(x)):
    total = totals[i]
    if total == 0:
        continue

    if up_counts[i] > 0:
        ax.text(
            x[i], up_counts[i] / 2,
            f"{int(up_counts[i])}",
            ha="center", va="center", color="black", fontsize=5
        )

    if down_counts[i] > 0:
        ax.text(
            x[i], up_counts[i] + down_counts[i] / 2,
            f"{int(down_counts[i])}",
            ha="center", va="center", color="black", fontsize=5
        )


    # if up_counts[i] > 0:
    #     ax.text(x[i], up_counts[i] / 2, f"{up_p:.0%}", ha="center", va="center", color="black", fontsize=9)
    # if down_counts[i] > 0:
    #     ax.text(x[i], up_counts[i] + down_counts[i] / 2, f"{down_p:.0%}", ha="center", va="center", color="black", fontsize=9)

    ax.text(x[i], total, f"{int(total)}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(output_path, f"Consistent_lfc{LFC_COL}_{today}.svg"), dpi = 300)
plt.show()


In [ ]:
target_genes = activity_candidate_genes_up_consistent + activity_candidate_genes_down_consistent
activity_candidate_consistent_df = activity_candidate_df[activity_candidate_df["gene"].isin(target_genes)]
activity_candidate_consistent_df

In [ ]:
# activity_candidate_consistent_df.to_csv(
#     os.path.join(
#         output_path,
#         f"activity_candidate_consistent_df_{today}.csv"
#     ),
#     index=False
# )

## 3. Permutation-based validation (shuffle test)

Finally, a permutation-based shuffle test was applied to disrupt the correspondence between gene identities and expression values. This procedure was used as a negative control to estimate the likelihood that the observed consistency arose by chance, providing an additional layer of validation for the identified activity selectivity–associated genes.

In [ ]:
# 1) 基因列表
genes_for_shuffle = activity_candidate_consistent_df['gene']
log2fc_for_shuffle = log2fc_df.reindex(genes_for_shuffle).dropna(how="all")

# 2) 观察统计量
obs_stat = log2fc_for_shuffle.median(axis=1)  # 每个gene的median

# 3) 置换（随机翻转每个样本列符号）
n_perm = 1000
rng = np.random.default_rng(42)

X = log2fc_for_shuffle.to_numpy()          # shape: (G, M)
G, M = X.shape
signs = rng.choice([-1, 1], size=(n_perm, M))  # shape: (P, M)

# perm_stats shape: (P, G)
perm_stats = np.median(X[None, :, :] * signs[:, None, :], axis=2)

# 4) 双尾阈值（5% / 95%）
thr_lo = np.percentile(perm_stats, 5, axis=0)
thr_hi = np.percentile(perm_stats, 95, axis=0)

obs = obs_stat.to_numpy()
pass_shuffle = (obs <= thr_lo) | (obs >= thr_hi)

pass_shuffle = pd.Series(pass_shuffle, index=obs_stat.index, name="pass_shuffle")
thr_df = pd.DataFrame({"thr_lo": thr_lo, "thr_hi": thr_hi}, index=obs_stat.index)


In [ ]:
result = pd.DataFrame({
    "obs_median_log2FC": obs_stat,
    "perm_thr_lo": thr_df["thr_lo"],
    "perm_thr_hi": thr_df["thr_hi"],
    "pass_shuffle": pass_shuffle
})

# 额外加一列，表示超出阈值的幅度（便于排序）
obs = result["obs_median_log2FC"]
result["exceed_margin"] = np.where(
    obs >= result["perm_thr_hi"], obs - result["perm_thr_hi"],
    np.where(obs <= result["perm_thr_lo"], result["perm_thr_lo"] - obs, 0.0)
)

result = result.sort_values(
    ["pass_shuffle", "exceed_margin", "obs_median_log2FC"],
    ascending=[False, False, False]
)

result.head(10)


In [ ]:
permutation_genes = result.index[result["pass_shuffle"]].tolist()
activity_candidate_final_df = activity_candidate_consistent_df[activity_candidate_consistent_df["gene"].isin(permutation_genes)]
activity_candidate_final_df

In [ ]:
activity_candidate_final_df.to_csv(
    os.path.join(
        output_path,
        f"activity_candidate_final_genes_{today}.csv"
    ),
    index=True
)

In [ ]:
activity_candidate_final_df

Mcf2l	-0.123873	0.123873	True
AC149090.1	-0.210688	0.210688	True
Kifc2	-0.254753	0.254753	True
Pisd-ps1	-0.416365	0.416365	True
L3mbtl1	-0.632368	0.632368	True
Adcyap1r1	-1.377872	1.377872	True
Vim	-1.511342	1.511342	True

In [ ]:
# Normalize in place
sc.pp.normalize_total(adata, target_sum=1e6)   # A higher target_sum enhances visual contrast

# Log-transform
sc.pp.log1p(adata)

In [ ]:
sc.pl.violin(
    adata_no_pc118,
    keys=['Gpr85'],
    groupby='condition'
)

In [ ]:
adata

In [ ]:
import scanpy as sc

genes = ["Gpr85","Gpr137c","Nlgn3","Ap2m1","Plat",
         "Gdi2","Irak1","Stmn4","Map2k3"]

order = ["silent","nonplace", "place" ]

adata.obs["condition"] = pd.Categorical(
    adata.obs["condition"],
    categories=order,
    ordered=True
)
sc.pl.heatmap(
    adata,
    var_names=genes,
    groupby="condition",     # place / nonplace / silent
    use_raw=False,           # 用 adata.X（通常是 log-normalized）
    layer=None,              # 如果你存了某个 layer，就填 layer="lognorm"
    standard_scale="var",    # 对每个基因做0-1标准化，便于比较模式（可选）
    swap_axes=True,          # 让 y=genes, x=condition（更像论文）
    dendrogram=False,
    figsize=(3, 4),
)

In [ ]:
adata_no_pc118 = adata[adata.obs["sample"].notna() & adata.obs["sample"].ne("PC118")].copy()
adata_no_pc118.obs["condition"].value_counts()

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

genes = ["Gpr85","Gpr137c","Nlgn3","Ap2m1","Plat",
         "Gdi2","Irak1","Stmn4","Map2k3"]

def make_mouse_condition_adata(adata, genes, sample_col="sample", condition_col="condition", layer=None):
    # 取表达矩阵（cells × genes）
    X = adata[:, genes].layers[layer] if layer is not None else adata[:, genes].X
    X = X.toarray() if sparse.issparse(X) else np.asarray(X)

    obs = adata.obs[[sample_col, condition_col]].copy()
    obs["mouse_cond"] = obs[sample_col].astype(str) + "|" + obs[condition_col].astype(str)

    # 对每个 mouse_cond 求均值（你也可以改成 median）
    dfX = pd.DataFrame(X, columns=genes, index=obs.index)
    meanX = dfX.groupby(obs["mouse_cond"]).mean()

    # 生成新的 AnnData
    ad = sc.AnnData(meanX.values)
    ad.var_names = genes
    ad.obs_names = meanX.index.astype(str)

    # 拆回 sample/condition
    ad.obs[sample_col] = [s.split("|")[0] for s in ad.obs_names]
    ad.obs[condition_col] = [s.split("|")[1] for s in ad.obs_names]

    return ad

adata_mc = make_mouse_condition_adata(adata_no_pc118, genes, sample_col="sample", condition_col="condition", layer=None)
order = ["silent","nonplace", "place" ]

adata_mc.obs["condition"] = pd.Categorical(
    adata_mc.obs["condition"],
    categories=order,
    ordered=True
)

sc.pl.heatmap(
    adata_mc,
    var_names=genes,
    groupby="condition",        # 仍然按 condition 排序
    use_raw=False,
    standard_scale="var",       # 推荐：看模式
    swap_axes=True,
    dendrogram=False,            # 可选：对 condition 聚类（通常不需要）
    figsize=(2, 2),
)

In [ ]:
adata.obs["sample"].value_counts()